Parsing raw data from text files
================================

Starting point for the project is the parsing of data files. Text data files contain the raw METAR data from a station.

```console
...
200501011300 METAR MRLB 011300Z 03004KT 9999 FEW040 23/20 A2990=
200501011400 METAR MRLB 011400Z 05004KT 9999 FEW040 27/20 A2992=
200501011500 METAR MRLB 011500Z 06016KT 9999 SCT040 30/20 A2994=
200501011600 METAR MRLB 011600Z 07014KT 9999 SCT040 31/20 A2993=
200501011700 METAR MRLB 011700Z 08016KT 9999 SCT040 32/20 A2992=
200501011800 METAR MRLB 011800Z 07014KT 9999 SCT040 32/20 A2990=
...
```

First we have to transform the data in something usable for other projects, this is, create CSV files.
The start point of this notebook is parsing a file and transform it into a CSV file, we took the 2005.txt file
for MRLB station.

As you can see, every line consist on a twelve digit date integer with format `%Y%m%d%H%M` and the METAR for that
date and time. We separate every line in a python `datetime` and the METAR as `str`. For this purpose we use the
[Pydantic](https://docs.pydantic.dev/latest/) `BaseModel` to configure an object with a defined structure. Then
a function called `parse_metar_line` makes the work.

Get a step behind in the directory tree
=======================================

In [1]:
%cd ..
%ls -la

/home/diego/development/python/metar-datasets
total 468
drwxr-xr-x. 1 diego diego    358 nov  4 19:21 ./
drwxr-xr-x. 1 diego diego    858 nov 25 14:48 ../
-rw-r--r--. 1 diego diego    148 feb  7  2024 .bumpversion.cfg
drwxr-xr-x. 1 diego diego     42 nov  4 18:43 data/
drwxr-xr-x. 1 diego diego     18 nov  4 19:07 docs/
drwxr-xr-x. 1 diego diego     72 nov  4 19:36 .dvc/
-rw-r--r--. 1 diego diego    139 nov  4 19:21 .dvcignore
drwxr-xr-x. 1 diego diego    144 dic 14 14:25 .git/
drwxr-xr-x. 1 diego diego     18 feb  7  2024 .github/
-rw-r--r--. 1 diego diego   1393 feb  7  2024 .gitignore
-rw-r--r--. 1 diego diego     96 feb  7  2024 .isort.cfg
-rwxr-xr-x. 1 diego diego   1053 nov  4 18:48 LICENSE*
-rwxr-xr-x. 1 diego diego   1446 nov  4 19:13 Makefile*
-rwxr-xr-x. 1 diego diego    149 feb  7  2024 mypy.ini*
drwxr-xr-x. 1 diego diego     70 nov 14 20:33 notebooks/
-rw-r--r--. 1 diego diego 427117 nov 15 15:27 poetry.lock
-rw-r--r--. 1 diego diego    808 nov 15 15:27 pyproject.toml
-rw-r

/home/diego/.cache/pypoetry/virtualenvs/metar-datasets-VqN08WBd-py3.11/lib/python3.11/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


Open a METAR's file
===================

A METAR's file consist of a text file that every line is a single METAR as seen before.

In [2]:
RAW_DATA_DIR = "./data/raw/mrlb/"

f = open(RAW_DATA_DIR + "2005.txt")

Preparing a model to parse every line of the METAR's file
=============================================================

In [3]:
from datetime import datetime

from pydantic import BaseModel

class ParsedMetarLine(BaseModel):
    date: datetime
    metar: str

def parse_metar_line(l: str) -> ParsedMetarLine:
    l = l.replace("=", "")
    l = l.strip()
    
    date = l[0:12]
    metar = l[13:]
    
    date_obj = datetime.strptime(date, "%Y%m%d%H%M")
    
    parsed_metar_line = ParsedMetarLine(date=date_obj, metar=metar)
    return parsed_metar_line

Creating a function to parse a entire METAR's file
==================================================

Once we have the code that manipulates the text line, we can parse the METAR with its correct date using the [parse_metar_to_dataframe](https://unidata.github.io/MetPy/latest/api/generated/metpy.io.parse_metar_to_dataframe.html) function from [MetPy](https://unidata.github.io/MetPy/latest/index.html) package, the function get a METAR text with its corresponding year and month as optional parameters, and returns a [Pandas](https://pandas.pydata.org/docs/index.html) `DataFrame`.

We can concatenate every METAR dataframe until get the entire file parsed.

In [4]:
import pandas as pd

from io import TextIOWrapper

from metpy.io import parse_metar_to_dataframe

def parse_metars_file(f: TextIOWrapper) -> pd.DataFrame:
    metars_df = pd.DataFrame()
    
    for line in f:
        if "NIL" in line:
            continue
        
        parsed_line = parse_metar_line(line)
        
        single_metar_df = parse_metar_to_dataframe(
            parsed_line.metar,
            year=parsed_line.date.year,
            month=parsed_line.date.month,
        )
        
        metars_df = pd.concat([metars_df, single_metar_df], ignore_index=True, )
    
    return metars_df

Parsing the METAR's file and getting a Pandas DataFrame (pandas.DataFrame)
===========================================================================

And here we get the `DataFrame` of METAR we are looking for.

In [5]:
metars_df = parse_metars_file(f)
metars_df.head()

,station_id,latitude,longitude,elevation,date_time,wind_direction,wind_speed,wind_gust,visibility,current_wx1,...,air_temperature,dew_point_temperature,altimeter,current_wx1_symbol,current_wx2_symbol,current_wx3_symbol,remarks,air_pressure_at_sea_level,eastward_wind,northward_wind
0,MRLB,10.6,-85.55,82,2005-01-01 01:00:00,80.0,15.0,NaN,9999.0,NaN,...,27.0,19.0,29.84,0,0,0,,1010.39,-14.772116,-2.604723
1,MRLB,10.6,-85.55,82,2005-01-01 13:00:00,30.0,4.0,NaN,9999.0,NaN,...,23.0,20.0,29.90,0,0,0,,1012.55,-2.000000,-3.464102
2,MRLB,10.6,-85.55,82,2005-01-01 14:00:00,50.0,4.0,NaN,9999.0,NaN,...,27.0,20.0,29.92,0,0,0,,1013.10,-3.064178,-2.571150
3,MRLB,10.6,-85.55,82,2005-01-01 15:00:00,60.0,16.0,NaN,9999.0,NaN,...,30.0,20.0,29.94,0,0,0,,1013.69,-13.856406,-8.000000
4,MRLB,10.6,-85.55,82,2005-01-01 16:00:00,70.0,14.0,NaN,9999.0,NaN,...,31.0,20.0,29.93,0,0,0,,1013.32,-13.155697,-4.788282


Coverting the `DataFrame` to CSV file
=====================================

Once the `DataFrame` is constructed, we use its `to_csv` method to save the data in our data folder.

In [6]:
import os

PROCESSED_DATA_DIR = "./data/processed/mrlb/metar/csv/"

try:
    os.makedirs(PROCESSED_DATA_DIR)
except FileExistsError:
    pass

metars_df.to_csv(PROCESSED_DATA_DIR + "metars.csv")